In [ ]:
import json
import pandas as pd

with open("../data/products.json", "r", encoding="utf-8") as f:
    products = json.load(f)
for p in products:
    ingred_str = p["clean_ingreds"].replace("'", '"')
    try:
        p["clean_ingreds"] = json.loads(ingred_str)
    except json.JSONDecodeError:
        p["clean_ingreds"] = []

In [2]:
for p in products:
    price_str = str(p.get("price", "0")).replace("£", "").strip()
    try:
        p["price"] = float(price_str)
    except:
        p["price"] = 0.0

In [3]:
with open("../data/users.json", "r", encoding="utf-8") as f:
    users = json.load(f)

severity_cols = ["Acne_Severity", "Dryness_Severity", "Pigmentation_Severity", "Aging_Severity", "Sensitivity_Severity"]
for u in users:
    for col in severity_cols:
        u[col] = float(u.get(col, 0))

In [4]:
ingredient_set = set()
for p in products:
    for ing in p["clean_ingreds"]:
        ing_clean = ing.lower().strip()
        ingredient_set.add(ing_clean)

all_ingredients = sorted(list(ingredient_set))

with open("../data/all_ingredients.txt", "w", encoding="utf-8") as f:
    for ing in all_ingredients:
        f.write(ing + "\n")

In [5]:
active_ingredients = {

# acne
"salicylic acid",
"benzoyl peroxide",
"niacinamide",
"azelaic acid",
"sulfur",
"zinc",

# hydration
"hyaluronic acid",
"glycerin",
"ceramide",
"ceramide np",
"ceramide ap",
"ceramide eop",
"squalane",
"panthenol",
"beta-glucan",

# pigmentation
"ascorbic acid",
"vitamin c",
"alpha-arbutin",
"kojic acid",
"niacinamide",

# anti-aging
"retinol",
"bakuchiol",
"peptides",
"adenosine",

# soothing
"allantoin",
"centella asiatica",
"green tea",
"chamomile",
"colloidal oatmeal",
"aloe vera",

# exfoliants
"glycolic acid",
"lactic acid",
"mandelic acid"
}

In [6]:
for p in products:
    
    p["active_ingredients"] = []

    for ing in p["clean_ingreds"]:
        ing = ing.lower().strip()

        if ing in active_ingredients:
            p["active_ingredients"].append(ing)

In [7]:
concern_to_ingredients = {

    "Acne_Severity": [
        "salicylic acid",
        "benzoyl peroxide",
        "niacinamide",
        "azelaic acid",
        "sulfur",
        "zinc"
    ],

    "Dryness_Severity": [
        "hyaluronic acid",
        "glycerin",
        "ceramide",
        "ceramide np",
        "squalane",
        "panthenol"
    ],

    "Pigmentation_Severity": [
        "ascorbic acid",
        "vitamin c",
        "alpha-arbutin",
        "kojic acid",
        "niacinamide"
    ],

    "Aging_Severity": [
        "retinol",
        "bakuchiol",
        "peptides",
        "adenosine"
    ],

    "Sensitivity_Severity": [
        "allantoin",
        "centella asiatica",
        "green tea",
        "chamomile",
        "colloidal oatmeal",
        "aloe vera",
        "panthenol"
    ]
}

In [8]:
def score_product(product, user):
    score = 0
    for concern, ingredients in concern_to_ingredients.items():
        severity = user.get(concern, 0)
        for ing in product["active_ingredients"]:
            if ing in ingredients:
                score += severity*2
    return score

In [9]:
budget_limits = {
    "Low": 20,
    "Medium": 50,
    "High": 200,
}

In [10]:
def rec_products(user, products, top_n=8):
    max_price = budget_limits.get(user["Budget_Level"], 200)
    scored_products = []
    for p in products:
        if p["price"] > max_price:
            continue
        score = score_product(p, user)
        if score > 0:
            scored_products.append((p, score))
    
    scored_products.sort(key=lambda x: x[1], reverse=True)
    return scored_products[:top_n]

In [11]:
user = users[0]

recs = rec_products(user, products)

print("User Profile:\n")
print(user)

print("\nRecommended Products:\n")

for product, score in recs:

    print("Product:", product["product_name"])
    print("Type:", product["product_type"])
    print("Price:", product["price"])
    print("Active Ingredients:", product["active_ingredients"])
    print("Score:", score)
    print("URL:", product["product_url"])
    print("-----------")

User Profile:

{'User_ID': '0', 'Skin_Type': 'Sensitive', 'Budget_Level': 'High', 'Acne_Severity': 0.0, 'Dryness_Severity': 0.0, 'Pigmentation_Severity': 0.0, 'Aging_Severity': 2.4, 'Sensitivity_Severity': 6.336962022905103}

Recommended Products:

Product: COSRX Advanced Snail 92 All in One Cream 100ml
Type: Moisturiser
Price: 27.0
Active Ingredients: ['panthenol', 'allantoin', 'adenosine']
Score: 30.147848091620414
URL: https://www.lookfantastic.com/cosrx-advanced-snail-92-all-in-one-cream-100ml/11401173.html
-----------
Product: Alpha-H Beauty Sleep Power Peel 50ml
Type: Peel
Price: 43.2
Active Ingredients: ['glycolic acid', 'glycerin', 'retinol', 'panthenol', 'allantoin']
Score: 30.147848091620414
URL: https://www.lookfantastic.com/alpha-h-beauty-sleep-power-peel-50ml/11282237.html
-----------
Product: Peter Thomas Roth Retinol Eye Care 15ml
Type: Eye Care
Price: 46.0
Active Ingredients: ['retinol', 'ascorbic acid', 'allantoin', 'panthenol', 'ceramide np']
Score: 30.147848091620414